In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
from pathlib import Path
import numpy as np  # linear algebra
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

In [ ]:
### cell 0 ###
benchmark_name = "environmental-vs-ai-startups-india-eda"
file = (
    Path(BENCHMARKS_TO_PATHS[benchmark_name])
    .parent
    / "input"
    / "indian-startup-recognized-by-dpiit"
    / "Startup_Counts_Across_India.csv"
)
# read into a cudf-backed DataFrame via pandas API
df = pd.read_csv(file)
factor = 3000
# revert to pd.concat to preserve original row order and duplicate indices exactly
df = pd.concat([df] * factor)
# mirror original call
df.info()

In [ ]:
### cell 1 ###

df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

In [ ]:
### cell 2 ###

### cell 2 (optimized for cudf)

# Industry sub-categories for environmental & AI startups
env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai  = ["AI", "Robotics", "Computer Vision"]
ea_cats = env + ai

# 1) one GPU-native isin + reset_index
df_ea = df[df["Industry"].isin(ea_cats)].reset_index(drop=True)

# 2) single boolean mask for ENV vs AI
mask_env = df_ea["Industry"].isin(env)

# 3) vectorized assignment without CPU map
#    default to "AI", then overwrite ENV where mask_env is True
df_ea["MainIndustry"] = "AI"
df_ea.loc[mask_env, "MainIndustry"] = "ENV"

# 4) stats with GPU-native ops
total     = int(df_ea.shape[0])
env_count = int(mask_env.sum())
ai_count  = total - env_count

print(
    f"A total of {total} startups were started in India between 2016 & 2022, "
    f"out of which {env_count} are environmental related & {ai_count} are AI startups."
)